# Aurora Embedding Comparison: Global vs Cropped ERA5

Extracts **encoder** and **backbone bottleneck** embeddings from
`AuroraSmallPretrained` for both global and regionally-cropped ERA5 data,
then compares them with **t-SNE** and **cosine similarity**.

**Run `era5_download_crop.ipynb` first** to produce the saved Batch files.

In [ ]:
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from torch.nn.functional import cosine_similarity

DATA_DIR = (
    Path(__file__).resolve().parent.parent / "data" / "era5" / "batches"
    if "__file__" in dir()
    else Path.cwd().parent / "data" / "era5" / "batches"
)
DATE = "2023-01-01"
PATCH_SIZE = 4

## 1. Load Saved Batches

In [ ]:
from aurora import Batch

global_batch  = Batch.from_netcdf(DATA_DIR / f"{DATE}-global.nc")
cropped_batch = Batch.from_netcdf(DATA_DIR / f"{DATE}-cropped.nc")

print(f"Global:  {global_batch.spatial_shape}")
print(f"Cropped: {cropped_batch.spatial_shape}")

## 2. Load Model & Register Hooks

In [ ]:
from aurora import AuroraSmallPretrained

model = AuroraSmallPretrained()
model.load_checkpoint("microsoft/aurora", "aurora-0.25-small-pretrained.ckpt")
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Device: {device}")

captured = {}


def _make_hook(name):
    def fn(module, inp, out):
        captured[name] = (out[0] if isinstance(out, tuple) else out).detach().cpu()
    return fn


model.encoder.register_forward_hook(_make_hook("encoder"))
model.backbone.encoder_layers[-1].register_forward_hook(_make_hook("bottleneck"))
print("Hooks registered on encoder and backbone bottleneck.")

## 3. Extract Embeddings

In [ ]:
def extract(batch):
    captured.clear()
    with torch.inference_mode():
        model(batch)
    return captured["encoder"].clone(), captured["bottleneck"].clone()


print("Running global batch...")
g_enc, g_bot = extract(global_batch)
print(f"  encoder   {g_enc.shape}")
print(f"  bottleneck {g_bot.shape}")

print("Running cropped batch...")
c_enc, c_bot = extract(cropped_batch)
print(f"  encoder   {c_enc.shape}")
print(f"  bottleneck {c_bot.shape}")

## 4. Compute Spatial Correspondence

Extract the sub-region of the **global** embeddings that corresponds
to the same geographic area as the cropped embeddings.

In [ ]:
L = model.encoder.latent_levels  # 4


def post_crop_hw(batch):
    h, w = batch.spatial_shape
    if h % PATCH_SIZE == 1:
        h -= 1
    return h, w


gH, gW = post_crop_hw(global_batch)
cH, cW = post_crop_hw(cropped_batch)

# Encoder-level patch grids
g_eH, g_eW = gH // PATCH_SIZE, gW // PATCH_SIZE
c_eH, c_eW = cH // PATCH_SIZE, cW // PATCH_SIZE

# Bottleneck patch grids (2× PatchMerging = 4× spatial reduction)
g_bH, g_bW = g_eH // 4, g_eW // 4
c_bH, c_bW = c_eH // 4, c_eW // 4

# Pixel offset of cropped region inside the global grid
g_lat0 = global_batch.metadata.lat[0].item()
c_lat0 = cropped_batch.metadata.lat[0].item()
c_lon0 = cropped_batch.metadata.lon[0].item()

lat_px = int(round((g_lat0 - c_lat0) / 0.25))
lon_px = int(round(c_lon0 / 0.25))

enc_lat0 = lat_px // PATCH_SIZE
enc_lon0 = lon_px // PATCH_SIZE
bot_lat0 = enc_lat0 // 4
bot_lon0 = enc_lon0 // 4

print(f"Encoder   – global ({g_eH}×{g_eW}), crop ({c_eH}×{c_eW}), offset ({enc_lat0},{enc_lon0})")
print(f"Bottleneck – global ({g_bH}×{g_bW}), crop ({c_bH}×{c_bW}), offset ({bot_lat0},{bot_lon0})")

In [ ]:
def slice_region(emb, full_hw, crop_hw, offset, L):
    """Reshape (1, N, D) → (L, H, W, D), then extract the crop window."""
    fH, fW = full_hw
    rH, rW = crop_hw
    y0, x0 = offset
    D = emb.shape[-1]
    grid = emb[0].reshape(L, fH, fW, D)
    return grid[:, y0 : y0 + rH, x0 : x0 + rW, :].reshape(-1, D)


# Encoder
g_enc_region = slice_region(g_enc, (g_eH, g_eW), (c_eH, c_eW), (enc_lat0, enc_lon0), L)
c_enc_flat   = c_enc[0]  # (N, D)

# Bottleneck
g_bot_region = slice_region(g_bot, (g_bH, g_bW), (c_bH, c_bW), (bot_lat0, bot_lon0), L)
c_bot_flat   = c_bot[0]

print(f"Encoder   – global region {g_enc_region.shape}, cropped {c_enc_flat.shape}")
print(f"Bottleneck – global region {g_bot_region.shape}, cropped {c_bot_flat.shape}")

## 5. t-SNE: Encoder Embeddings

In [ ]:
N_enc = g_enc_region.shape[0]
enc_stack = torch.cat([g_enc_region, c_enc_flat[:N_enc]], dim=0).float().numpy()
enc_labels = np.array(["global"] * N_enc + ["cropped"] * N_enc)

tsne_enc = TSNE(n_components=2, perplexity=30, random_state=42, init="pca", learning_rate="auto")
proj_enc = tsne_enc.fit_transform(enc_stack)

fig, ax = plt.subplots(figsize=(8, 6))
for lab, col in [("global", "#2196F3"), ("cropped", "#FF5722")]:
    m = enc_labels == lab
    ax.scatter(proj_enc[m, 0], proj_enc[m, 1], s=4, alpha=0.5, c=col, label=lab)
ax.legend()
ax.set_title("t-SNE — Encoder Embeddings (same geographic region)")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
plt.tight_layout()
plt.show()

## 6. t-SNE: Bottleneck Embeddings

In [ ]:
N_bot = g_bot_region.shape[0]
bot_stack = torch.cat([g_bot_region, c_bot_flat[:N_bot]], dim=0).float().numpy()
bot_labels = np.array(["global"] * N_bot + ["cropped"] * N_bot)

perp = min(30, max(5, N_bot - 1))
tsne_bot = TSNE(n_components=2, perplexity=perp, random_state=42, init="pca", learning_rate="auto")
proj_bot = tsne_bot.fit_transform(bot_stack)

fig, ax = plt.subplots(figsize=(8, 6))
for lab, col in [("global", "#2196F3"), ("cropped", "#FF5722")]:
    m = bot_labels == lab
    ax.scatter(proj_bot[m, 0], proj_bot[m, 1], s=20, alpha=0.7, c=col, label=lab)
ax.legend()
ax.set_title("t-SNE — Bottleneck Embeddings (same geographic region)")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
plt.tight_layout()
plt.show()

## 7. Quantitative Comparison

In [ ]:
enc_cos = cosine_similarity(g_enc_region, c_enc_flat[:N_enc], dim=-1)
bot_cos = cosine_similarity(g_bot_region, c_bot_flat[:N_bot], dim=-1)

enc_mse = ((g_enc_region - c_enc_flat[:N_enc]) ** 2).mean().item()
bot_mse = ((g_bot_region - c_bot_flat[:N_bot]) ** 2).mean().item()

print("Per-patch cosine similarity (global-region vs cropped):")
print(f"  Encoder    – mean {enc_cos.mean():.4f}  std {enc_cos.std():.4f}  min {enc_cos.min():.4f}")
print(f"  Bottleneck – mean {bot_cos.mean():.4f}  std {bot_cos.std():.4f}  min {bot_cos.min():.4f}")
print(f"\nMSE:  encoder {enc_mse:.6f}   bottleneck {bot_mse:.6f}")

## 8. Spatial Cosine-Similarity Heatmaps

Per-patch cosine similarity between global (same region) and cropped encoder embeddings,
shown for each latent level.

In [ ]:
enc_cos_spatial = enc_cos.reshape(L, c_eH, c_eW)

fig, axes = plt.subplots(1, L, figsize=(4 * L, 3.5))
if L == 1:
    axes = [axes]
for lev in range(L):
    im = axes[lev].imshow(
        enc_cos_spatial[lev].numpy(),
        vmin=0.8, vmax=1.0, cmap="RdYlGn", aspect="auto",
    )
    axes[lev].set_title(f"Latent Level {lev}")
    axes[lev].set_xlabel("Lon patch")
    if lev == 0:
        axes[lev].set_ylabel("Lat patch")
fig.colorbar(im, ax=axes, label="Cosine Similarity", shrink=0.8)
fig.suptitle("Encoder: Per-Patch Cosine Similarity (Global vs Cropped)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
bot_cos_spatial = bot_cos.reshape(L, c_bH, c_bW)

fig, axes = plt.subplots(1, L, figsize=(4 * L, 3.5))
if L == 1:
    axes = [axes]
for lev in range(L):
    im = axes[lev].imshow(
        bot_cos_spatial[lev].numpy(),
        vmin=0.5, vmax=1.0, cmap="RdYlGn", aspect="auto",
    )
    axes[lev].set_title(f"Latent Level {lev}")
    axes[lev].set_xlabel("Lon patch")
    if lev == 0:
        axes[lev].set_ylabel("Lat patch")
fig.colorbar(im, ax=axes, label="Cosine Similarity", shrink=0.8)
fig.suptitle("Bottleneck: Per-Patch Cosine Similarity (Global vs Cropped)", y=1.02)
plt.tight_layout()
plt.show()